# Connecting Claude to the GitHub MCP Server

So far we've given Claude *our own* tools (function-calling) and Anthropic's *server-side* tools (web search). In this notebook we add a third, very powerful option: **MCP (Model Context Protocol)**.

**What is MCP?** MCP is an open standard that lets an AI application talk to external tool servers in a uniform way. Instead of writing custom integration code for every service (GitHub, Slack, a database, your internal API...), a service exposes an **MCP server**, and any **MCP client** can connect to it and use its tools.

**Why does this matter for *our* application?** Claude's Messages API has a built-in **MCP connector**. That means *Anthropic's servers* act as the MCP client on our behalf — we just point Claude at an MCP server URL, and Claude can discover and call that server's tools automatically inside a single API request. No extra infrastructure, no third-party orchestration library.

In this example we connect to the **GitHub MCP server** so Claude can answer questions about *our own* GitHub repository (`mariapanneerrajan/intro_to_ai_engineering_with_claude_api`) — commits, issues, files, and more — live, without us writing any GitHub API code.

## Prerequisites

1. **An Anthropic API key** — already loaded from `.env` like in the previous notebooks.
2. **A GitHub Personal Access Token (PAT)** — the GitHub MCP server needs to authenticate as *you* to read your repos.
   - Create one at **GitHub → Settings → Developer settings → Personal access tokens**.
   - A fine-grained token with **read-only** access to the repo you want to ask about is plenty.
   - Store it in your `.env` file as `GITHUB_TOKEN=...` — **never hard-code a token in the notebook.**

Minimal dependencies: we only use the official `anthropic` SDK plus `python-dotenv` to load secrets — the same two packages used throughout this workshop.

## Imports & Client Setup

In [ ]:
import os
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

# Anthropic() automatically reads ANTHROPIC_API_KEY from the environment.
client = Anthropic()

# Haiku is fast and cost-effective for this demo. The MCP connector works with all models.
model = "claude-haiku-4-5"

# Read the GitHub PAT from the environment — keep secrets out of the notebook.
github_token = os.environ["GITHUB_TOKEN"]

## Describe the MCP Server

To use the MCP connector we provide **two matching pieces** in the API call:

1. **`mcp_servers`** — *where* the server is and how to authenticate. GitHub's hosted MCP server lives at `https://api.githubcopilot.com/mcp/`. We pass our PAT as the `authorization_token`.
2. **`tools` with an `mcp_toolset`** — this *enables* the tools from that server for Claude to use. It references the server by the `name` we gave it.

Both must be supplied together. The MCP connector also currently requires a **beta header** (`mcp-client-2025-11-20`), which is why we call it through `client.beta.messages`.

In [ ]:
# 1) Point Claude at the GitHub MCP server and authenticate as us.
mcp_servers = [
    {
        "type": "url",
        "url": "https://api.githubcopilot.com/mcp/",
        "name": "github",  # our local label for this server
        "authorization_token": github_token,
    }
]

# 2) Enable the tools exposed by the "github" server.
tools = [
    {
        "type": "mcp_toolset",
        "mcp_server_name": "github",  # must match the name above
    }
]

## Ask Claude About Our Repo

Now we just ask a natural-language question. Behind the scenes Claude will:
1. Discover the tools the GitHub MCP server offers,
2. Decide which ones to call (e.g. list commits, read files, search issues),
3. Call them through Anthropic's MCP connector, and
4. Summarize the results back to us — all within this single API request.

In [ ]:
messages = [
    {
        "role": "user",
        "content": (
            "What are my 3 public GitHub repos with recent activity?"
        ),
    }
]

response = client.beta.messages.create(
    model=model,
    max_tokens=1024,
    betas=["mcp-client-2025-11-20"],  # enables the MCP connector
    mcp_servers=mcp_servers,
    tools=tools,
    messages=messages,
)

In [ ]:
print(response)

## Inspect the Response

The response `content` is a list of blocks. With MCP you'll typically see a mix of:
- **`mcp_tool_use`** blocks — the calls Claude made to the GitHub server,
- **`mcp_tool_result`** blocks — what the server returned,
- **`text`** blocks — Claude's final, human-readable answer.

Let's print the block types to see the tool calls, then print just the final text answer.

In [ ]:
# See the sequence of blocks (tool calls, results, and final text).
for block in response.content:
    print("-", block.type)

In [ ]:
# Print Claude's final natural-language answer about our repo.
for block in response.content:
    if block.type == "text":
        print(block.text)

## Why This Is Powerful

Notice what we did **not** have to do:
- We wrote **no GitHub API calls** — no pagination, no endpoint URLs, no JSON parsing.
- We defined **no tool schemas** ourselves — the MCP server advertises its own tools.
- We ran **no extra servers or orchestration libraries** — Anthropic's MCP connector acts as the client.

By adding a single `mcp_servers` entry, our application instantly gained the ability to *read and reason over a live external system*. The same pattern works for any MCP server — Sentry, Slack, a database, or your own internal MCP server — turning Claude into a hub that can act across all your tools.

### Try it yourself
- Ask about **open issues or pull requests** in the repo.
- Ask Claude to **read a specific file** (e.g. `08_tool_use.ipynb`) and explain it.
- Connect a **second** MCP server and ask a question that spans both.